In [ ]:
"import sys\n","sys.path.insert(0, '../src')\n","\n","import torch\n","from torch.utils.data import DataLoader\n","import pytorch_lightning as pl\n","from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping\n","from pytorch_lightning.loggers import TensorBoardLogger\n","\n","from config import load_config\n","from dataset_coco import COCOInstanceDataset\n","from models import Mask2Former\n","from lightning_module import SegmentationModule"

In [ ]:
"config = load_config('../configs/config_coco_mask2former_instance.json')\n","\n","DATA_ROOT = config['data']['root_dir']\n","TRAIN_SPLIT = config['data']['split']\n","VAL_SPLIT = config['data']['val_split']\n","TEST_SPLIT = config['data']['test_split']\n","TARGET_SIZE = tuple(config['data']['target_size'])\n","BATCH_SIZE = config['data']['batch_size']\n","NUM_WORKERS = config['data']['num_workers']\n","MIN_AREA = config['data']['min_area']\n","MODE = config['data']['mode']\n","NUM_CLASSES = config['model']['num_classes']\n","\n","MAX_EPOCHS = config['training']['max_epochs']\n","LEARNING_RATE = config['training']['learning_rate']\n","\n","NUM_QUERIES = config['model']['num_queries']\n","HIDDEN_DIM = config['model']['hidden_dim']\n","NHEADS = config['model']['nheads']\n","NUM_DECODER_LAYERS = config['model']['num_decoder_layers']\n","\n","print(f\"Model: Mask2Former (Hybrid)\")\n","print(f\"Classes: {NUM_CLASSES}\")\n"

In [ ]:
"def collate_fn(batch):\n","    images = torch.stack([item[0] for item in batch])\n","    masks = [item[1] for item in batch]\n","    labels = [item[2] for item in batch]\n","    return images, masks, labels\n","\n","train_dataset = COCOInstanceDataset(\n","    root_dir=DATA_ROOT,\n","    split=TRAIN_SPLIT,\n","    target_size=TARGET_SIZE,\n","    min_area=MIN_AREA,\n","    mode=MODE\n",")\n","\n","val_dataset = COCOInstanceDataset(\n","    root_dir=DATA_ROOT,\n","    split=VAL_SPLIT,\n","    target_size=TARGET_SIZE,\n","    min_area=MIN_AREA,\n","    mode=MODE\n",")\n","\n","test_dataset = COCOInstanceDataset(\n","    root_dir=DATA_ROOT,\n","    split=TEST_SPLIT,\n","    target_size=TARGET_SIZE,\n","    min_area=MIN_AREA,\n","    mode=MODE\n",")\n","\n","print(f'Train samples: {len(train_dataset)}')\n","print(f'Val samples: {len(val_dataset)}')\n","print(f'Test samples: {len(test_dataset)}')\n"

In [ ]:
"train_loader = DataLoader(\n","    train_dataset,\n","    batch_size=BATCH_SIZE,\n","    shuffle=True,\n","    num_workers=NUM_WORKERS,\n","    pin_memory=True,\n","    collate_fn=collate_fn\n",")\n","\n","val_loader = DataLoader(\n","    val_dataset,\n","    batch_size=BATCH_SIZE,\n","    shuffle=False,\n","    num_workers=NUM_WORKERS,\n","    pin_memory=True,\n","    collate_fn=collate_fn\n",")\n","\n","test_loader = DataLoader(\n","    test_dataset,\n","    batch_size=BATCH_SIZE,\n","    shuffle=False,\n","    num_workers=NUM_WORKERS,\n","    pin_memory=True,\n","    collate_fn=collate_fn\n",")\n","\n","test_loader = DataLoader(\n","    test_dataset,\n","    batch_size=BATCH_SIZE,\n","    shuffle=False,\n","    num_workers=NUM_WORKERS,\n","    pin_memory=True,\n","    collate_fn=collate_fn\n",")\n"

In [ ]:
"import matplotlib.pyplot as plt\n","import numpy as np\n","\n","sample_images, sample_masks, sample_labels = next(iter(train_loader))\n","\n","mean = np.array([0.485, 0.456, 0.406])\n","std = np.array([0.229, 0.224, 0.225])\n","\n","fig, axes = plt.subplots(2, 2, figsize=(12, 12))\n","axes = axes.flatten()\n","\n","for i in range(min(2, len(sample_images))):\n","    img = sample_images[i].cpu().numpy().transpose(1, 2, 0)\n","    img = img * std + mean\n","    img = np.clip(img, 0, 1)\n","    \n","    axes[i*2].imshow(img)\n","    axes[i*2].set_title(f'Image {i+1}')\n","    axes[i*2].axis('off')\n","    \n","    combined_mask = np.zeros(sample_masks[i].shape[1:], dtype=np.int32)\n","    for j, mask in enumerate(sample_masks[i]):\n","        combined_mask[mask.numpy() > 0] = j + 1\n","    \n","    axes[i*2+1].imshow(combined_mask, cmap='tab20')\n","    axes[i*2+1].set_title(f'Instances: {len(sample_masks[i])}')\n","    axes[i*2+1].axis('off')\n","\n","plt.tight_layout()\n","plt.show()\n","\n","print(f\"\\nBatch info:\")\n","print(f\"Images: {sample_images.shape}\")\n","print(f\"Sample 0: {len(sample_masks[0])} instances, labels: {sample_labels[0].tolist()[:10]}...\")\n"

In [ ]:
"model = Mask2Former(\n","    num_classes=NUM_CLASSES,\n","    num_queries=NUM_QUERIES,\n","    emb_dim=HIDDEN_DIM,\n","    nhead=NHEADS,\n","    nlayers=NUM_DECODER_LAYERS\n",")\n","\n","model = SegmentationModule(\n","    model=model,\n","    num_classes=NUM_CLASSES,\n","    learning_rate=LEARNING_RATE,\n","    weight_decay=config['training']['weight_decay'],\n","    optimizer='adamw'\n",")\n","\n","print(f\"Mask2Former model created for instance segmentation\")\n","print(f\"  Transformer queries: {NUM_QUERIES}\")\n","print(f\"  Embedding dimension: {HIDDEN_DIM}\")\n","print(f\"  Attention heads: {NHEADS}\")\n","print(f\"  Decoder layers: {NUM_DECODER_LAYERS}\")\n"

In [ ]:
"checkpoint_callback = ModelCheckpoint(\n","    dirpath=config['training']['checkpoint_dirpath'],\n","    filename=config['training']['checkpoint_filename'],\n","    save_top_k=config['training']['checkpoint_save_top_k'],\n","    monitor='val_loss',\n","    mode='min'\n",")\n","\n","early_stopping = EarlyStopping(\n","    monitor='val_loss',\n","    patience=config['training']['early_stopping_patience'],\n","    mode='min'\n",")\n","\n","logger = TensorBoardLogger(\n","    save_dir=config['training']['log_dir'],\n","    name=config['training']['experiment_name']\n",")\n"

In [ ]:
"trainer = pl.Trainer(\n","    max_epochs=100,\n","    callbacks=[checkpoint_callback, early_stopping],\n","    logger=logger,\n","    accelerator=config['hardware']['accelerator'],\n","    devices=config['hardware']['devices'],\n","    log_every_n_steps=config['training']['log_every_n_steps']\n",")\n"

In [ ]:
"trainer.fit(model, train_loader, val_loader)\n"

"# Testing: Semantic Segmentation (mIoU)"

In [ ]:
"from tqdm.notebook import tqdm\n","from torchmetrics import JaccardIndex\n","\n","model.eval()\n","test_iou = JaccardIndex(\n","    task='multiclass',\n","    num_classes=NUM_CLASSES + 1,\n","    average='macro',\n","    ignore_index=0\n",").to(model.device)\n","\n","with torch.no_grad():\n","    for batch in tqdm(test_loader, desc='Semantic mIoU'):\n","        images, instance_masks, instance_labels = batch\n","        images = images.to(model.device)\n","        \n","        outputs = model(images)\n","        preds = model.model.postprocess(outputs, mode='semantic')\n","        \n","        semantic_targets = model._instance_to_semantic(\n","            instance_masks, instance_labels, device=model.device\n","        )\n","        \n","        test_iou.update(preds, semantic_targets)\n","\n","semantic_miou = test_iou.compute()\n","print(f'\\nSemantic Segmentation mIoU: {semantic_miou:.4f}')\n"

"# Testing: Instance Segmentation (mAP)"

In [ ]:
"from torchmetrics.detection import MeanAveragePrecision\n","\n","test_map = MeanAveragePrecision(iou_type='segm').to(model.device)\n","\n","with torch.no_grad():\n","    for batch in tqdm(test_loader, desc='Instance mAP'):\n","        images, instance_masks, instance_labels = batch\n","        images = images.to(model.device)\n","        \n","        outputs = model(images)\n","        preds = model.model.postprocess(outputs, mode='instance')\n","        \n","        preds_coco, targets_coco = [], []\n","        for b in range(len(images)):\n","            if len(preds[b]['classes']) > 0:\n","                pred_dict = {\n","                    'masks': (preds[b]['masks'] > 0.5).cpu().to(torch.uint8),\n","                    'labels': preds[b]['classes'].cpu(),\n","                    'scores': preds[b]['scores'].cpu()\n","                }\n","            else:\n","                pred_dict = {\n","                    'masks': torch.zeros((0, 360, 480), dtype=torch.uint8),\n","                    'labels': torch.tensor([], dtype=torch.long),\n","                    'scores': torch.tensor([], dtype=torch.float32)\n","                }\n","            preds_coco.append(pred_dict)\n","            \n","            valid_idx = instance_labels[b] > 0\n","            if valid_idx.any():\n","                gt_dict = {\n","                    'masks': (instance_masks[b][valid_idx] > 0.5).cpu().to(torch.uint8),\n","                    'labels': instance_labels[b][valid_idx].cpu()\n","                }\n","            else:\n","                gt_dict = {\n","                    'masks': torch.zeros((0, 360, 480), dtype=torch.uint8),\n","                    'labels': torch.tensor([], dtype=torch.long)\n","                }\n","            targets_coco.append(gt_dict)\n","        \n","        test_map.update(preds_coco, targets_coco)\n","\n","map_metrics = test_map.compute()\n","print(f\"\\nInstance Segmentation Metrics:\")\n","print(f\"  mAP (IoU=0.50:0.95): {map_metrics['map']:.4f}\")\n","print(f\"  mAP@50:              {map_metrics['map_50']:.4f}\")\n","print(f\"  mAP@75:              {map_metrics['map_75']:.4f}\")\n"

"# Visualization of Predictions"